# 05 - CV comparison charts (per block), bootstrap CI

Reads `finetune_shared*.pt` (notebooks 02/03) and draws, for tiny and base, the NN-head and
LR-head 5-fold CV accuracy per fine-tuned depth (last 1/2/3 blocks) with 95% bootstrap CIs,
against the frozen (0.831) and handcrafted (0.872) reference lines. Writes to results_withseed/.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
OUT='../results_withseed'
labels=np.load('../data/preprocessed/labels.npy'); N=len(labels)
FROZEN_CV, HANDCRAFT_CV = 0.831, 0.872

def boot_ci(pred, n_boot=2000, seed=123):
    rng=np.random.default_rng(seed); idx=np.arange(N); vals=[]
    for _ in range(n_boot):
        b=rng.choice(idx,N,replace=True); vals.append(accuracy_score(labels[b],pred[b]))
    lo,hi=np.percentile(vals,[2.5,97.5]); return accuracy_score(labels,pred), lo, hi

def cv_chart(cv, head, key, model_name, fname):
    accs,elo,ehi=[],[],[]
    for nf in [10,9,8]:
        a,lo,hi=boot_ci(cv[nf][key]); accs.append(a); elo.append(a-lo); ehi.append(hi-a)
    x=np.arange(3); fig,ax=plt.subplots(figsize=(8,5))
    b=ax.bar(x,accs,0.55,yerr=[elo,ehi],capsize=6,color='#B07AA1',error_kw=dict(lw=1.2,ecolor='#444'))
    ax.bar_label(b,fmt='%.3f',padding=10,fontsize=10)
    ax.axhline(FROZEN_CV,color='green',ls='--',lw=1.3,label='Frozen + LR (0.831)')
    ax.axhline(HANDCRAFT_CV,color='red',ls=':',lw=1.3,label='Handcrafted + LR (0.872)')
    ax.set_xticks(x); ax.set_xticklabels(['Last 1 block','Last 2 blocks','Last 3 blocks'])
    ax.set_ylabel('Accuracy'); ax.set_ylim(0.5,1.0)
    ax.set_title(f'{head} - Fine-Tuned {model_name} (5-fold CV, 95% bootstrap CI)')
    ax.legend(fontsize=9,loc='upper right'); ax.grid(axis='y',alpha=0.3); ax.set_axisbelow(True)
    plt.tight_layout(); plt.savefig(os.path.join(OUT,fname),dpi=150); plt.show()
    print('Saved',fname,'| accs',[round(a,3) for a in accs])

for tag,suffix,mname in [('tiny','','BrainOmni-tiny'),('base','_base','BrainOmni-base')]:
    fn=f'{OUT}/finetune_shared{suffix}.pt'
    if not os.path.exists(fn):
        print(f'skip {tag}: {fn} not found (run notebooks 02/03 first)'); continue
    cv=torch.load(fn,map_location='cpu',weights_only=False)['cv']
    cv_chart(cv,'Neural Net Head','nn_pred',mname,f'nn_comparison{suffix}_boot.png')
    cv_chart(cv,'Logistic Regression','lr_pred',mname,f'lr_comparison{suffix}_boot.png')